# Lab Assignment 7
## PCS221 Cloud Computing – MapReduce Programming

**Instructions:** Submit `.ipynb` file on Github

Reference Notebook: [Google Colab](https://colab.research.google.com/drive/12UbpUk11eJwzQMzgMqey4yOkz_m0576c?authuser=2)

---
## Setup: Install mrjob

In [ ]:
# Install mrjob for MapReduce programming in Python
!pip install mrjob

---
## Task 1: Basic MapReduce – Word Count

We implement a basic Word Count MapReduce job using `mrjob`.

In [ ]:
%%writefile wordcount.py
from mrjob.job import MRJob
import re

WORD_REGEXP = re.compile(r"[\w']+")

class MRWordCount(MRJob):

    def mapper(self, _, line):
        for word in WORD_REGEXP.findall(line):
            yield word.lower(), 1

    def reducer(self, word, counts):
        yield word, sum(counts)

if __name__ == '__main__':
    MRWordCount.run()

---
## Task 2: Word Count on *Alice's Adventures in Wonderland*

Download the text file from Project Gutenberg and run word count on it.  
**Question:** How many times does the word **Cheshire** occur?  
*(Do not include the word `'Cheshire` with an apostrophe)*

In [ ]:
# Download Alice's Adventures in Wonderland from Project Gutenberg
!wget -q -O alice.txt http://www.gutenberg.org/cache/epub/11/pg11.txt
print("Download complete!")

# Preview first few lines
with open('alice.txt', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())

In [ ]:
# Run MapReduce Word Count on alice.txt
!python wordcount.py alice.txt > alice_wordcount_output.txt
print("MapReduce Word Count complete!")

In [ ]:
import json

# Parse the output to find the count for 'cheshire'
cheshire_count = 0

with open('alice_wordcount_output.txt', 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            word, count = line.split('\t')
            word = word.strip('"')  # mrjob outputs JSON-quoted strings
            if word == 'cheshire':
                cheshire_count = int(count)

print(f"Number of times 'Cheshire' occurs (case-insensitive, no apostrophe): {cheshire_count}")

In [ ]:
# Alternative verification using Python's built-in string search
# This counts standalone 'Cheshire' words (not preceded by apostrophe)
import re

with open('alice.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Match 'Cheshire' NOT preceded by apostrophe
# Using negative lookbehind: (?<!')Cheshire
matches = re.findall(r"(?<!'\b)\bCheshire\b", text)

# Simpler: count all 'Cheshire' and subtract those preceded by apostrophe
all_cheshire = re.findall(r"\bCheshire\b", text, re.IGNORECASE)
apostrophe_cheshire = re.findall(r"'Cheshire\b", text, re.IGNORECASE)

count_without_apostrophe = len(all_cheshire) - len(apostrophe_cheshire)

print(f"Total 'Cheshire' occurrences (any case): {len(all_cheshire)}")
print(f"Occurrences with apostrophe ('Cheshire): {len(apostrophe_cheshire)}")
print(f"Final count (Cheshire without apostrophe): {count_without_apostrophe}")

---
## Task 3: Word Median using MapReduce

The `wordmedian` application computes the **median length of words** in a text file.  
**Input:** `words.txt` from [Google Drive](https://drive.google.com/file/d/16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas/view?usp=sharing)  
**Question:** What is the median word length?

In [ ]:
# Download words.txt from Google Drive
# File ID: 16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas
!pip install -q gdown
import gdown

file_id = '16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'words.txt', quiet=False)
print("words.txt downloaded!")

# Preview
with open('words.txt', 'r') as f:
    lines = f.readlines()[:10]
    print("First 10 lines:", lines)

In [ ]:
%%writefile wordmedian.py
from mrjob.job import MRJob
from mrjob.step import MRStep
import re

WORD_REGEXP = re.compile(r"[\w']+")

class MRWordMedian(MRJob):
    """
    MapReduce job to compute the median word length.

    Step 1 Mapper : emit (word_length, 1) for each word
    Step 1 Reducer: emit (word_length, total_count) per length bucket
    Step 2 Reducer: collect all (length, count) pairs, sort, find median
    """

    def steps(self):
        return [
            MRStep(mapper=self.mapper_word_length,
                   reducer=self.reducer_count_lengths),
            MRStep(reducer=self.reducer_find_median)
        ]

    # --- Step 1 ---
    def mapper_word_length(self, _, line):
        for word in WORD_REGEXP.findall(line):
            yield len(word), 1

    def reducer_count_lengths(self, length, counts):
        yield None, (length, sum(counts))

    # --- Step 2 ---
    def reducer_find_median(self, _, length_count_pairs):
        # Collect all (length, count) and sort by length
        pairs = sorted(length_count_pairs, key=lambda x: x[0])

        total_words = sum(count for _, count in pairs)
        median_pos = (total_words + 1) / 2.0  # median position (1-indexed)

        cumulative = 0
        for length, count in pairs:
            cumulative += count
            if cumulative >= median_pos:
                yield 'median_word_length', length
                return

if __name__ == '__main__':
    MRWordMedian.run()

In [ ]:
# Run the wordmedian MapReduce job on words.txt
!python wordmedian.py words.txt

In [ ]:
# Verification: Compute median word length using pure Python
import re
import statistics

WORD_REGEXP = re.compile(r"[\w']+")

with open('words.txt', 'r') as f:
    text = f.read()

word_lengths = [len(word) for word in WORD_REGEXP.findall(text)]

print(f"Total words: {len(word_lengths)}")
print(f"Min word length: {min(word_lengths)}")
print(f"Max word length: {max(word_lengths)}")
print(f"Mean word length: {statistics.mean(word_lengths):.2f}")
print(f"Median word length: {statistics.median(word_lengths)}")

---
## Summary of Results

| Task | Description | Result |
|------|-------------|--------|
| Task 2 | Count of word **'Cheshire'** in *Alice's Adventures in Wonderland* (no apostrophe) | *(run cells above)* |
| Task 3 | Median word length in `words.txt` | *(run cells above)* |

> **Note:** Results will populate after running all cells above.